In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder \
    .appName("RideSharing_Silver") \
    .getOrCreate()

In [4]:
base_path = "/content/drive/MyDrive/Ride_Sharing"

In [5]:
drivers = spark.read.parquet(
    f"{base_path}/data/bronze/drivers"
)

trips = spark.read.parquet(
    f"{base_path}/data/bronze/trips"
)

logs = spark.read.parquet(
    f"{base_path}/data/bronze/trip_logs"
)

In [6]:
drivers = drivers.dropDuplicates()

trips = trips.dropDuplicates()

logs = logs.dropDuplicates()

In [7]:
drivers = drivers.dropna()

trips = trips.dropna()

logs = logs.dropna()

In [8]:
drivers.show(5)

trips.show(5)

logs.show(5)

+---------+---------+---------+------+
|driver_id|     name|     city|rating|
+---------+---------+---------+------+
|       14| Rohit_14|     Pune|   3.7|
|      147|Karan_147|     Pune|   3.7|
|       83| Rohit_83|Bangalore|   4.1|
|      141|Arjun_141|   Mumbai|   3.9|
|      119|Sneha_119|   Mumbai|   3.6|
+---------+---------+---------+------+
only showing top 5 rows
+-------+---------+---------------+---------------+-----------+-----------+-----------+
|trip_id|driver_id|pickup_location|  drop_location|distance_km|fare_amount|trip_status|
+-------+---------+---------------+---------------+-----------+-----------+-----------+
|     14|       99|        IT Park|        IT Park|      17.96|     232.49|  Completed|
|     72|       31|    City Center|           Mall|       16.9|        0.0|  Cancelled|
|     15|       85|        IT Park|        IT Park|      14.48|     157.98|  Completed|
|     88|       44|    City Center|Railway Station|       6.03|        0.0|  Cancelled|
|     23|

In [12]:
driver_trip = trips.join(
    drivers,
    on="driver_id",
    how="inner"
)

driver_trip.show(5)

+---------+-------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+
|driver_id|trip_id|pickup_location|  drop_location|distance_km|fare_amount|trip_status|    name|     city|rating|
+---------+-------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+
|       99|     14|        IT Park|        IT Park|      17.96|     232.49|  Completed|Rahul_99|    Delhi|   4.2|
|       31|     72|    City Center|           Mall|       16.9|        0.0|  Cancelled| Neha_31|   Mumbai|   4.6|
|       85|     15|        IT Park|        IT Park|      14.48|     157.98|  Completed|Karan_85|   Mumbai|   5.0|
|       44|     88|    City Center|Railway Station|       6.03|        0.0|  Cancelled|Karan_44|     Pune|   4.6|
|       65|     23|           Mall|Railway Station|       21.2|     239.39|  Completed| Amit_65|Bangalore|   3.6|
+---------+-------+---------------+---------------+-----------+-----------+-----------+-

In [13]:
silver = driver_trip.join(
    logs,
    on="trip_id",
    how="inner"
)

silver.show(5)

+-------+---------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|trip_id|driver_id|pickup_location|  drop_location|distance_km|fare_amount|trip_status|    name|     city|rating|log_id|         start_time|           end_time|delay_minutes|cancellation_flag|
+-------+---------+---------------+---------------+-----------+-----------+-----------+--------+---------+------+------+-------------------+-------------------+-------------+-----------------+
|     14|       99|        IT Park|        IT Park|      17.96|     232.49|  Completed|Rahul_99|    Delhi|   4.2|    14|2025-01-04 14:23:00|2025-01-04 14:32:00|           20|                0|
|     15|       85|        IT Park|        IT Park|      14.48|     157.98|  Completed|Karan_85|   Mumbai|   5.0|    15|2025-01-02 08:39:00|2025-01-02 09:23:00|           19|                0|
|     23|       65|           Mall|

In [14]:
from pyspark.sql.functions import unix_timestamp, col

silver = silver.withColumn(
    "trip_duration",
    (unix_timestamp(col("end_time")) - unix_timestamp(col("start_time"))) / 60
)

silver.select(
    "trip_id",
    "start_time",
    "end_time",
    "trip_duration"
).show(5)

+-------+-------------------+-------------------+-------------+
|trip_id|         start_time|           end_time|trip_duration|
+-------+-------------------+-------------------+-------------+
|     14|2025-01-04 14:23:00|2025-01-04 14:32:00|          9.0|
|     15|2025-01-02 08:39:00|2025-01-02 09:23:00|         44.0|
|     23|2025-01-06 11:01:00|2025-01-06 11:12:00|         11.0|
|    130|2025-01-06 18:40:00|2025-01-06 19:25:00|         45.0|
|     99|2025-01-04 05:53:00|2025-01-04 06:11:00|         18.0|
+-------+-------------------+-------------------+-------------+
only showing top 5 rows


In [15]:
from pyspark.sql.functions import when

silver = silver.withColumn(
    "completion_status",
    when(col("cancellation_flag") == 1, "Cancelled")
    .otherwise("Completed")
)

silver.select(
    "trip_id",
    "cancellation_flag",
    "completion_status"
).show(5)

+-------+-----------------+-----------------+
|trip_id|cancellation_flag|completion_status|
+-------+-----------------+-----------------+
|     14|                0|        Completed|
|     15|                0|        Completed|
|     23|                0|        Completed|
|    130|                0|        Completed|
|     99|                0|        Completed|
+-------+-----------------+-----------------+
only showing top 5 rows


In [16]:
silver = silver.withColumn(
    "driver_performance",
    when(col("rating") >= 4.5, "Excellent")
    .when(col("rating") >= 4.0, "Good")
    .otherwise("Average")
)

silver.select(
    "driver_id",
    "name",
    "rating",
    "driver_performance"
).show(5)

+---------+--------+------+------------------+
|driver_id|    name|rating|driver_performance|
+---------+--------+------+------------------+
|       99|Rahul_99|   4.2|              Good|
|       85|Karan_85|   5.0|         Excellent|
|       65| Amit_65|   3.6|           Average|
|       80|Rohit_80|   4.8|         Excellent|
|       94| Amit_94|   3.6|           Average|
+---------+--------+------+------------------+
only showing top 5 rows


In [17]:
silver.printSchema()

silver.show(10, truncate=False)

root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- log_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: integer (nullable = true)
 |-- cancellation_flag: integer (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- completion_status: string (nullable = false)
 |-- driver_performance: string (nullable = false)

+-------+---------+---------------+---------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+-------------+-----

In [19]:
silver.write.mode("overwrite").parquet(
    f"{base_path}/data/silver/silver_data"
)

print("Silver Layer Created Successfully!")

Silver Layer Created Successfully!


In [20]:
silver_data = spark.read.parquet(
    f"{base_path}/data/silver/silver_data"
)

silver_data.show(10, truncate=False)

+-------+---------+---------------+---------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+-------------+-----------------+------------------+
|trip_id|driver_id|pickup_location|drop_location  |distance_km|fare_amount|trip_status|name     |city     |rating|log_id|start_time         |end_time           |delay_minutes|cancellation_flag|trip_duration|completion_status|driver_performance|
+-------+---------+---------------+---------------+-----------+-----------+-----------+---------+---------+------+------+-------------------+-------------------+-------------+-----------------+-------------+-----------------+------------------+
|14     |99       |IT Park        |IT Park        |17.96      |232.49     |Completed  |Rahul_99 |Delhi    |4.2   |14    |2025-01-04 14:23:00|2025-01-04 14:32:00|20           |0                |9.0          |Completed        |Good              |
|15     |85       |I